# Notebook 03 — Content-Based Recommender
**Member 2** | Uses TF-IDF on movie genres to recommend similar movies.

In [ ]:
import pandas as pd
import numpy as np
import pickle
import os
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print('Libraries loaded.')

## 1. Load Data

In [ ]:
movies = pd.read_csv('../data/processed/movies_clean.csv')
ratings = pd.read_csv('../data/processed/ratings_clean.csv')

print(f'Movies: {movies.shape}')
print(f'Ratings: {ratings.shape}')
movies.head()

## 2. Build TF-IDF Matrix on Genres

In [ ]:
# Detect genre column
genre_col = 'genres' if 'genres' in movies.columns else movies.columns[2]
print(f'Using genre column: {genre_col}')

movies['genre_str'] = movies[genre_col].fillna('unknown').astype(str)

tfidf = TfidfVectorizer(token_pattern=r'[^|,\s]+')
tfidf_matrix = tfidf.fit_transform(movies['genre_str'])

print(f'TF-IDF matrix shape: {tfidf_matrix.shape}')
print(f'Vocabulary size: {len(tfidf.vocabulary_)}')

## 3. Compute Cosine Similarity

In [ ]:
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)
print(f'Cosine similarity matrix: {cosine_sim.shape}')

# Index map
movie_id_col = 'movieId' if 'movieId' in movies.columns else movies.columns[0]
title_col = 'title' if 'title' in movies.columns else movies.columns[1]

indices = pd.Series(movies.index, index=movies[movie_id_col]).drop_duplicates()
print('Index map created.')

## 4. Content-Based Recommender Function

In [ ]:
def get_content_recommendations(movie_id, n=10):
    """Return top-n similar movies by content (genre TF-IDF)."""
    if movie_id not in indices:
        return pd.DataFrame()
    idx = indices[movie_id]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = [s for s in sim_scores if s[0] != idx][:n]
    movie_indices = [s[0] for s in sim_scores]
    result = movies.iloc[movie_indices][[movie_id_col, title_col, genre_col]].copy()
    result['similarity_score'] = [round(s[1], 4) for s in sim_scores]
    return result

# Quick test
sample_id = movies[movie_id_col].iloc[0]
recs = get_content_recommendations(sample_id, n=5)
print(f'Recommendations for movie_id={sample_id}:')
print(recs)

## 5. Evaluate on Sample Users

In [ ]:
user_col = 'userId' if 'userId' in ratings.columns else ratings.columns[0]
item_col = 'movieId' if 'movieId' in ratings.columns else ratings.columns[1]
rating_col = 'rating' if 'rating' in ratings.columns else ratings.columns[2]

sample_users = ratings[user_col].unique()[:20]
coverage_scores = []

for user in sample_users:
    user_movies = ratings[ratings[user_col] == user][item_col].tolist()
    if not user_movies:
        continue
    seed = user_movies[0]
    recs = get_content_recommendations(seed, n=10)
    if not recs.empty:
        overlap = len(set(recs[movie_id_col]) & set(user_movies))
        coverage_scores.append(overlap / len(user_movies))

avg_coverage = np.mean(coverage_scores) if coverage_scores else 0
print(f'Avg coverage across {len(sample_users)} users: {avg_coverage:.4f}')

## 6. Save Model

In [ ]:
os.makedirs('../models', exist_ok=True)

content_model = {
    'tfidf': tfidf,
    'tfidf_matrix': tfidf_matrix,
    'cosine_sim': cosine_sim,
    'movies': movies,
    'indices': indices,
    'movie_id_col': movie_id_col,
    'title_col': title_col,
    'genre_col': genre_col
}

with open('../models/content_model.pkl', 'wb') as f:
    pickle.dump(content_model, f)

print('content_model.pkl saved to ../models/')
print(f'Total movies in model: {len(movies)}')